![Figura](attachment:image.png)

In [ ]:
import os
from getpass import getpass
from urllib.parse import urlencode

import pandas as pd
import requests

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 50)

## 1. Uma função auxiliar para requisições GET

Método básico para fazer uma requisição

```python
resposta = requests.get(url,params,headers,timeout)
```

Praa facilitar as próximas etapas, vamos criar uma função para:

1. enviar a requisição;
2. definir um tempo máximo de espera;
3. verificar o código de status;
4. gerar uma exceção quando a resposta HTTP indicar erro.

O parâmetro `timeout` evita que o notebook aguarde indefinidamente por um servidor que não responde.

In [ ]:
def requisicao_get(url, params=None, headers=None, timeout=20):
    resposta = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=timeout
    )

    print("Status HTTP:", resposta.status_code)
    resposta.raise_for_status()

    return resposta

# 2. APIs com autenticação

Agora estudaremos serviços que exigem uma credencial para os exemplos propostos.

Nesta aula, a expressão **API com autenticação** será usada para indicar uma integração em que a requisição depende de uma credencial, como:

- API Key;
- access token;
- OAuth 2.0.

Isso não significa que todos os dados fornecidos pelo serviço sejam necessariamente privados. O ponto central é que o servidor precisa identificar ou autorizar o cliente antes de processar a operação utilizada no exemplo.

## 2.1. Nunca escreva segredos diretamente no notebook

Evite:

```python
TOKEN = "meu_token_real"
```

Um notebook pode ser:

- enviado a outra pessoa;
- publicado no GitHub;
- armazenado em nuvem;
- incluído em backups;
- exibido durante uma aula.

Nesta aula, utilizaremos `getpass()` para digitação temporária. Em projetos reais, prefira mecanismos apropriados de gerenciamento de segredos ou variáveis de ambiente.

# 3. API do Mercado Livre: configuração, OAuth e busca de produtos

Nesta seção, a configuração será feita **na mesma ordem em que o código será construído**. A finalidade é reduzir erros de autenticação e tornar o procedimento reproduzível em sala de aula.

Ao final, faremos uma busca por **Smart TV 70** no catálogo do Mercado Livre e transformaremos os produtos retornados em um `DataFrame`.

Documentação oficial consultada:

- criação da aplicação: https://developers.mercadolivre.com.br/pt_br/crie-uma-aplicacao-no-mercado-livre
- autenticação e autorização: https://developers.mercadolivre.com.br/autenticacao-e-autorizacao
- buscador de produtos: https://developers.mercadolivre.com.br/buscador-de-produtos

> A interface do DevCenter pode mudar. Os nomes exatos dos campos podem sofrer pequenas alterações, mas os conceitos de `Client ID`, `Client Secret`, `Redirect URI`, escopos e OAuth permanecem.


## 3.1. Acesso o DevCenter e clique em Criar nova aplicação

![Figura](attachment:image.png)

## 3.2. Preencha os dados e insira um logotipo qualquer.

![Figura](attachment:image-2.png)

## 3.3. Adicione uma URI

![Figura](attachment:image-3.png)

## 3.4. O que será configurado no DevCenter

Para esta aula, configure uma aplicação **somente para leitura**. Não precisamos publicar produtos, alterar anúncios ou receber notificações.

Preencha os campos da aplicação da seguinte forma:

| Campo no Mercado Livre | Configuração didática sugerida | Finalidade |
|---|---|---|
| **Nome** | `Aula API Mercado Livre` | Identificar a aplicação |
| **Nome curto** | um identificador simples e único | Identificação interna/URL da aplicação |
| **Descrição** | `Aplicação didática para consulta de produtos via API` | Texto apresentado no processo de autorização |
| **Logo** | opcional, conforme a interface permitir | Identificação visual |
| **Redirect URI** | `https://example.com` | Receber o retorno com o `code` |
| **PKCE** | desabilitado nesta primeira aula | Evita adicionar uma segunda camada ao fluxo didático |
| **Escopo Leitura** | habilitado | Necessário para chamadas `GET` |
| **Escopo Escrita** | desabilitado | Não será utilizado |
| **Tópicos/Notificações** | nenhum | Não precisamos receber eventos de Orders, Items, Catalog etc. |
| **URL de notificações** | não configurar | Só é necessária quando tópicos são selecionados |

### Por que usar `https://example.com`?

O Mercado Livre exige uma URI de redirecionamento com HTTPS. Para a aula, podemos utilizar uma URL de teste. Depois da autorização, observe a **barra de endereço do navegador**: o Mercado Livre acrescentará o parâmetro `code` à URL.

Exemplo conceitual:

```text
https://example.com/?code=TG-XXXXXXXXXXXX&state=YYYY
```

Mesmo que a página de destino não seja utilizada pela aplicação, o que nos interessa didaticamente é copiar o valor de `code` exibido na URL.

**Regra crítica:** use exatamente a mesma Redirect URI em três lugares:

1. cadastro da aplicação;
2. URL de autorização;
3. requisição que troca o `code` pelo token.

## 3.5. Resultado esperado
![Figura](attachment:image-4.png)

## 3.6. Depois de salvar a aplicação: identifique as credenciais

Após criar a aplicação, localize no DevCenter:

- **Client ID / APP ID**: identifica publicamente sua aplicação;
- **Client Secret / Secret Key**: segredo da aplicação;
- **Redirect URI**: endereço cadastrado anteriormente.

Nesta aula, o `Client Secret` nunca será escrito diretamente no notebook. Ele será solicitado por `getpass()`.

## 4. Construir o a URL de busca
## 4.1. registrar somente o Client ID e a Redirect URI

Começamos com os dois valores que **não são o token**. Nesta etapa ainda não faremos nenhuma requisição à API.


In [ ]:
app_id_ml = input("APP ID / Client ID do Mercado Livre: ").strip()
redirect_uri_ml = input("Redirect URI cadastrada [https://example.com]: ").strip()

if not redirect_uri_ml: redirect_uri_ml = "https://example.com"

print("APP ID informado:", app_id_ml)
print("Redirect URI:", redirect_uri_ml)


APP ID informado: 7628479356342648
Redirect URI: https://example.com


## 4.2. Etapa 2: construir a URL de autorização

A URL de autorização é formada por um **endpoint fixo** e **parâmetros variáveis**.

Para o Brasil:

```text
https://auth.mercadolivre.com.br/authorization
```

Parâmetros utilizados:

- `response_type=code`: solicita um código de autorização;
- `client_id`: APP ID da aplicação;
- `redirect_uri`: deve coincidir exatamente com o cadastro;
- `state`: valor aleatório usado para associar a resposta à solicitação iniciada.

Em vez de montar a URL manualmente, utilizaremos `urlencode()`. Isso evita erros com caracteres especiais.


In [ ]:
import secrets

state_ml = secrets.token_urlsafe(16)

parametros_autorizacao_ml = {
    "response_type": "code",
    "client_id": app_id_ml,
    "redirect_uri": redirect_uri_ml,
    "state": state_ml
}

url_autorizacao_ml = ("https://auth.mercadolivre.com.br/authorization?" + urlencode(parametros_autorizacao_ml))

print("Abra a URL abaixo no navegador:\n")
print(url_autorizacao_ml)
print("\nState gerado:", state_ml)


Abra a URL abaixo no navegador:

https://auth.mercadolivre.com.br/authorization?response_type=code&client_id=7628479356342648&redirect_uri=https%3A%2F%2Fexample.com&state=rpxNvuhujIxc5330oiBrwg

State gerado: rpxNvuhujIxc5330oiBrwg


## 4.3. Acesse o link gerado e a autorize a aplicação
![Figura](attachment:image.png)

## 4.4. autorizar no navegador e copiar o `code`

Agora o processo sai temporariamente do Python.

1. copie a URL produzida pela célula anterior;
2. abra-a no navegador;
3. faça login com a conta proprietária da aplicação;
4. autorize a aplicação;
5. aguarde o redirecionamento;
6. observe a URL final na barra de endereço;
7. copie somente o valor depois de `code=` e antes de `&state=`.

Exemplo:

```text
https://example.com/?code=TG-ABC123&state=XYZ456
```

Nesse exemplo:

```text
code  = TG-ABC123
state = XYZ456
```

O `code` é temporário e de uso único. Ele **ainda não é o access token**.


## 4.5. Trocar o `code` pelo access token

A troca é feita com uma requisição `POST` para:

```text
https://api.mercadolibre.com/oauth/token
```

Agora precisamos do `Client Secret`, por isso ele será digitado com `getpass()`.

> Se ocorrer `invalid_grant`, verifique primeiro: `code` já utilizado/expirado, Redirect URI diferente da cadastrada ou `code` pertencente a outra tentativa de autorização.


In [ ]:
client_secret_ml = getpass("Client Secret do Mercado Livre: ")
authorization_code_ml = getpass("Authorization code recebido: ")

url_token_ml = "https://api.mercadolibre.com/oauth/token"

dados_token_ml = {
    "grant_type": "authorization_code",
    "client_id": app_id_ml,
    "client_secret": client_secret_ml,
    "code": authorization_code_ml,
    "redirect_uri": redirect_uri_ml
}

resposta_token_ml = requests.post(
    url_token_ml,
    data=dados_token_ml,
    headers={
        "accept": "application/json",
        "content-type": "application/x-www-form-urlencoded"
    },
    timeout=20
)

print("Status HTTP:", resposta_token_ml.status_code)
resposta_token_ml.raise_for_status()

token_data_ml = resposta_token_ml.json()

print("Access token recebido:", "access_token" in token_data_ml)
print("Tipo:", token_data_ml.get("token_type"))
print("Expira em segundos:", token_data_ml.get("expires_in"))
print("Escopos:", token_data_ml.get("scope"))


Status HTTP: 200
Access token recebido: True
Tipo: Bearer
Expira em segundos: 21600
Escopos: read urn:global:admin:info:/read-only urn:global:admin:info:/read-write urn:global:admin:oauth:/read-only urn:global:admin:oauth:/read-write urn:global:admin:users:/read-only urn:ml:all:comunication:/read-only urn:ml:all:publish-sync:/read-only urn:ml:mktp:ads:/read-only urn:ml:mktp:comunication:/read-only urn:ml:mktp:invoices:/read-only urn:ml:mktp:metrics:/read-only urn:ml:mktp:offers:/read-only urn:ml:mktp:orders-shipments:/read-only urn:ml:mktp:publish-sync:/read-only write


## 4.6. Preparar o header de autenticação

A partir deste ponto, não precisamos colocar o token na URL. O Mercado Livre recomenda enviá-lo no header HTTP:

```text
Authorization: Bearer ACCESS_TOKEN
```

Essa configuração poderá ser reutilizada em todas as próximas chamadas autenticadas.


In [ ]:
access_token_ml = token_data_ml["access_token"]

headers_ml = {"Authorization": f"Bearer {access_token_ml}"}

print("Header preparado. O token não será exibido.")


Header preparado. O token não será exibido.


## 5. Teste mínimo de autenticação

Antes de construir uma consulta maior, fazemos uma chamada pequena a `/users/me`.

Objetivo desta célula: responder apenas à pergunta **"o token funciona?"**.


In [ ]:
resposta_usuario_ml = requisicao_get("https://api.mercadolibre.com/users/me", headers=headers_ml)

usuario_ml = resposta_usuario_ml.json()

print("Usuário autenticado:", usuario_ml.get("nickname"))
print("Site:", usuario_ml.get("site_id"))
print("ID:", usuario_ml.get("id"))


Status HTTP: 200
Usuário autenticado: LG20260914182200596
Site: MLB
ID: 3691497624


## 6. Buscar produtos do catálogo

Agora utilizaremos um endpoint adequado para pesquisa textual de **produtos de catálogo**:

```text
GET https://api.mercadolibre.com/products/search
```

Parâmetros principais:

- `site_id=MLB`: Brasil;
- `status=active`: produtos ativos;
- `q`: texto de busca;
- `limit`: número de resultados;
- `offset`: posição inicial para paginação.

Vamos pesquisar:

```text
Smart TV 70
```

Observe que o retorno não é uma tabela. A resposta possui metadados de paginação e uma lista em `results`.


In [ ]:
url_busca_ml = "https://api.mercadolibre.com/products/search"

parametros_busca_ml = {
    "site_id": "MLB",
    "status": "active",
    "q": "Smart TV 70",
    "limit": 20,
    "offset": 0
}

resposta_busca_ml = requisicao_get(
    url_busca_ml,
    params=parametros_busca_ml,
    headers=headers_ml
)

busca_ml = resposta_busca_ml.json()

print("Termo pesquisado:", busca_ml.get("keywords"))
print("Total informado pela API:", busca_ml.get("paging", {}).get("total"))
print("Resultados nesta página:", len(busca_ml.get("results", [])))


Status HTTP: 200
Termo pesquisado: Smart TV 70
Total informado pela API: 10000
Resultados nesta página: 20


## 6.1. Inspecionar a estrutura antes de criar o DataFrame

Antes de usar `pd.json_normalize()`, examine as chaves do primeiro produto. Essa prática evita selecionar colunas que não existem.


In [ ]:
resultados_ml = busca_ml.get("results", [])

if resultados_ml:
    print("Campos do primeiro produto:")
    print(list(resultados_ml[0].keys()))
else:
    print("Nenhum produto retornado para a consulta.")


Campos do primeiro produto:
['id', 'catalog_product_id', 'domain_id', 'name', 'parent_id', 'settings', 'children_ids', 'attributes', 'tags', 'status', 'short_description', 'pictures', 'authority_types', 'date_created', 'last_updated', 'quality_type', 'product_standard', 'search_type']


## 6.2. Transformar os resultados em um DataFrame

A lista `results` é a parte da resposta que corresponde ao conjunto de produtos. Como alguns campos são aninhados, utilizaremos `pd.json_normalize()`.

Neste primeiro DataFrame manteremos os atributos de nível principal. Depois criaremos colunas específicas como marca e modelo.


In [ ]:
df_produtos_ml = pd.json_normalize(resultados_ml)

print("Dimensão:", df_produtos_ml.shape)
df_produtos_ml.head()


Dimensão: (20, 21)


,id,catalog_product_id,domain_id,name,parent_id,children_ids,...,search_type,settings.content,settings.listing_strategy,settings.exclusive,short_description.type,short_description.content
0,MLB74613202,MLB74613202,MLB-TV_ANTENNAS,Antena Tdt para Smart TV Smart TV,MLB74613198,[],...,KEYWORD,fixed,catalog_required,False,plaintext,
1,MLB52789883,MLB52789883,MLB-TELEVISIONS,"Smart TV de 50"" Motorola Smart TV Motorola Sma...",NaN,[],...,KEYWORD,fixed,catalog_required,False,plaintext,
2,MLB63066047,MLB63066047,MLB-TELEVISIONS,Smart TV Samsung 75 Qled Q6 Smart TV (2025),NaN,[],...,KEYWORD,fixed,catalog_required,False,plaintext,
3,MLB36185154,MLB36185154,MLB-TELEVISIONS,Smart Tv LG Smart Tv 75ur8750psa Uhd 4k 75,NaN,[],...,KEYWORD,fixed,catalog_required,False,plaintext,
4,MLB62407410,MLB62407410,MLB-TELEVISIONS,Smart TV Hisense de 70 polegadas,NaN,[],...,KEYWORD,fixed,catalog_required,False,plaintext,


## 6.4. Extrair atributos como marca, modelo e tamanho da tela

No retorno do catálogo, características técnicas costumam aparecer dentro de `attributes`, uma lista de objetos.

Em vez de manter essa lista inteira em uma célula, criaremos uma função para procurar um atributo pelo seu `id`.

IDs frequentemente úteis para televisores incluem `BRAND`, `MODEL` e atributos relacionados ao tamanho da tela. Como a estrutura pode variar entre produtos, o código utiliza uma lista de alternativas para o tamanho.


In [ ]:
def valor_atributo(lista_atributos, ids_procurados):
    if not isinstance(lista_atributos, list):
        return None

    ids_procurados = set(ids_procurados)

    for atributo in lista_atributos:
        if atributo.get("id") in ids_procurados:
            return atributo.get("value_name")

    return None

if not df_produtos_ml.empty:
    df_produtos_ml["marca"] = df_produtos_ml["attributes"].apply(
        lambda x: valor_atributo(x, ["BRAND"])
    )
    df_produtos_ml["modelo"] = df_produtos_ml["attributes"].apply(
        lambda x: valor_atributo(x, ["MODEL"])
    )
    df_produtos_ml["tamanho_tela"] = df_produtos_ml["attributes"].apply(
        lambda x: valor_atributo(
            x,
            ["SCREEN_SIZE", "DISPLAY_SIZE", "TV_SIZE"]
        )
    )

    colunas_ml = [
        "id", "name", "status", "domain_id",
        "marca", "modelo", "tamanho_tela"
    ]

    df_produtos_ml_base = df_produtos_ml[
        [c for c in colunas_ml if c in df_produtos_ml.columns]
    ].copy()
else:
    df_produtos_ml_base = pd.DataFrame()

df_produtos_ml_base


,id,name,status,domain_id,marca,modelo,tamanho_tela
0,MLB74613202,Antena Tdt para Smart TV Smart TV,active,MLB-TV_ANTENNAS,Distronik,SB-3002,NaN
1,MLB52789883,"Smart TV de 50"" Motorola Smart TV Motorola Sma...",active,MLB-TELEVISIONS,Motorola,"Motorola Smart TV 50 50""","50 """
2,MLB63066047,Smart TV Samsung 75 Qled Q6 Smart TV (2025),active,MLB-TELEVISIONS,Samsung,Q6,"75 """
3,MLB36185154,Smart Tv LG Smart Tv 75ur8750psa Uhd 4k 75,active,MLB-TELEVISIONS,LG,75UR8750PSA,"75 """
4,MLB62407410,Smart TV Hisense de 70 polegadas,active,MLB-TELEVISIONS,Hisense,70A6,"70 """
5,MLB35086039,Smart TV Led Samsung 70 Un70Cu7000GCfv,active,MLB-TELEVISIONS,Samsung,UN70CU7000GCFV,"70 """
6,MLB62599860,Samsung Smart Tv 85 Uhd 4k + Samsung Smart Tv 50,active,MLB-TELEVISIONS,Samsung,UN85U8100FGXZD,"85 """
7,MLB24658978,"Smart TV Vizio M65q7-j01 Quantum 65"" Smart TV 4K",active,MLB-TELEVISIONS,Vizio,M65Q7-J01,"65 """
8,MLB62868674,Smart TV 5.0 Qled Q7f Vision Ai Smart TV (2025),active,MLB-TELEVISIONS,Samsung,QN50Q7FAAGX,"50 """
9,MLB21558719,Smart TV Hisense 60 4k Uhd Smart TV Hdr Roku T...,active,MLB-TELEVISIONS,Hisense,60R6E,"60 """


## 6.5. Obter detalhes dos produtos encontrados

A busca fornece uma lista resumida. Para aprofundar a coleta, podemos consultar cada `product_id` em:

```text
GET https://api.mercadolibre.com/products/{PRODUCT_ID}
```

O detalhe pode incluir `permalink`, `family_name`, atributos técnicos, imagens e informações da oferta vencedora (`buy_box_winner`) quando disponíveis.

Para não gerar muitas requisições em sala, consultaremos apenas os cinco primeiros produtos.


In [ ]:
detalhes_produtos_ml = []

for product_id in df_produtos_ml_base.get("id", pd.Series(dtype=str)).head(5):
    resposta = requisicao_get(
        f"https://api.mercadolibre.com/products/{product_id}",
        headers=headers_ml
    )
    detalhes_produtos_ml.append(resposta.json())

print("Produtos detalhados:", len(detalhes_produtos_ml))


Status HTTP: 200
Status HTTP: 200
Status HTTP: 200
Status HTTP: 200
Status HTTP: 200
Produtos detalhados: 5


In [ ]:
linhas_detalhes_ml = []

for produto in detalhes_produtos_ml:
    vencedor = produto.get("buy_box_winner") or {}

    linhas_detalhes_ml.append({
        "product_id": produto.get("id"),
        "nome": produto.get("name"),
        "familia": produto.get("family_name"),
        "status": produto.get("status"),
        "dominio": produto.get("domain_id"),
        "preco_oferta_vencedora": vencedor.get("price"),
        "moeda": vencedor.get("currency_id"),
        "frete_gratis": (vencedor.get("shipping") or {}).get("free_shipping"),
        "permalink": produto.get("permalink")
    })

df_detalhes_ml = pd.DataFrame(linhas_detalhes_ml)
df_detalhes_ml


,product_id,nome,familia,status,dominio,preco_oferta_vencedora,moeda,frete_gratis,permalink
0,MLB74613202,Antena Tdt para Smart TV Smart TV,Antena de TV Distronik SB-3002 para interno,active,MLB-TV_ANTENNAS,None,None,None,
1,MLB52789883,"Smart TV de 50"" Motorola Smart TV Motorola Sma...","Smart TV de 50"" Motorola Smart TV Motorola Sma...",active,MLB-TELEVISIONS,None,None,None,
2,MLB63066047,Smart TV Samsung 75 Qled Q6 Smart TV (2025),Smart TV Samsung 75 Qled Q6 Smart TV (2025),active,MLB-TELEVISIONS,None,None,None,
3,MLB36185154,Smart Tv LG Smart Tv 75ur8750psa Uhd 4k 75,Smart Tv LG Smart Tv 75ur8750psa Uhd 4k 75,active,MLB-TELEVISIONS,None,None,None,
4,MLB62407410,Smart TV Hisense de 70 polegadas,Smart TV Hisense de 70 polegadas,active,MLB-TELEVISIONS,None,None,None,


## 6.6. Paginação da busca

O Mercado Livre utiliza `offset` e `limit` neste recurso. A ideia é deslocar a janela de resultados:

```text
offset=0   → primeira página
offset=20  → segunda página
offset=40  → terceira página
```

A função abaixo reúne várias páginas em um único DataFrame.


In [ ]:
def buscar_produtos_ml(termo, paginas=3, limite=20):
    produtos = []

    for pagina in range(paginas):
        params = {
            "site_id": "MLB",
            "status": "active",
            "q": termo,
            "limit": limite,
            "offset": pagina * limite
        }

        resposta = requests.get(
            "https://api.mercadolibre.com/products/search",
            params=params,
            headers=headers_ml,
            timeout=20
        )
        resposta.raise_for_status()

        lote = resposta.json().get("results", [])
        produtos.extend(lote)

        if len(lote) < limite:
            break

    return pd.json_normalize(produtos)


In [ ]:
# Exemplo:
df_tv_70_multiplas_paginas = buscar_produtos_ml("Smart TV 70", paginas=3)
df_tv_70_multiplas_paginas.head(30)

,id,catalog_product_id,domain_id,name,parent_id,children_ids,...,settings.content,settings.listing_strategy,settings.exclusive,short_description.type,short_description.content,enhanced_content_template_id
0,MLB74613202,MLB74613202,MLB-TV_ANTENNAS,Antena Tdt para Smart TV Smart TV,MLB74613198,[],...,fixed,catalog_required,False,plaintext,,NaN
1,MLB52789883,MLB52789883,MLB-TELEVISIONS,"Smart TV de 50"" Motorola Smart TV Motorola Sma...",NaN,[],...,fixed,catalog_required,False,plaintext,,NaN
2,MLB63066047,MLB63066047,MLB-TELEVISIONS,Smart TV Samsung 75 Qled Q6 Smart TV (2025),NaN,[],...,fixed,catalog_required,False,plaintext,,NaN
3,MLB36185154,MLB36185154,MLB-TELEVISIONS,Smart Tv LG Smart Tv 75ur8750psa Uhd 4k 75,NaN,[],...,fixed,catalog_required,False,plaintext,,NaN
4,MLB62407410,MLB62407410,MLB-TELEVISIONS,Smart TV Hisense de 70 polegadas,NaN,[],...,fixed,catalog_required,False,plaintext,,NaN
5,MLB35086039,MLB35086039,MLB-TELEVISIONS,Smart TV Led Samsung 70 Un70Cu7000GCfv,NaN,[],...,fixed,catalog_required,False,plaintext,A Samsung é reconhecida mundialmente como uma ...,NaN
6,MLB62599860,MLB62599860,MLB-TELEVISIONS,Samsung Smart Tv 85 Uhd 4k + Samsung Smart Tv 50,NaN,[],...,fixed,catalog_required,False,plaintext,Descubra a incrível experiência de entretenime...,NaN
7,MLB24658978,MLB24658978,MLB-TELEVISIONS,"Smart TV Vizio M65q7-j01 Quantum 65"" Smart TV 4K",NaN,[],...,fixed,catalog_required,False,plaintext,"Prático e de apenas 65, ele permitirá que você...",NaN
8,MLB62868674,MLB62868674,MLB-TELEVISIONS,Smart TV 5.0 Qled Q7f Vision Ai Smart TV (2025),NaN,[],...,fixed,catalog_required,False,plaintext,,NaN
9,MLB21558719,MLB21558719,MLB-TELEVISIONS,Smart TV Hisense 60 4k Uhd Smart TV Hdr Roku T...,NaN,[],...,fixed,catalog_required,False,plaintext,A Hisense é a marca de televisão número 1 na C...,NaN


## 7. Diagnóstico rápido de erros do Mercado Livre

| Situação | Causa provável | Verificação |
|---|---|---|
| `redirect_uri` inválida | URI diferente entre cadastro e código | compare caractere por caractere |
| `invalid_grant` | `code` expirado, usado ou associado a outra tentativa | gere um novo `code` |
| `invalid_client` | Client ID ou Client Secret incorretos | confira o DevCenter |
| `401` | token ausente, inválido ou expirado | refaça autenticação/renovação |
| `403` | escopo/permissão insuficiente ou restrição da aplicação | revise escopos e conta |
| `429` | excesso de requisições | reduza frequência e implemente espera |


# 21. Exercícios

Os exercícios abaixo devem ser resolvidos reutilizando as células anteriores, sem escrever credenciais diretamente no notebook.

### Nível 1: busca simples

Pesquise `notebook gamer` no catálogo brasileiro e construa um `DataFrame` com pelo menos:

- `id`;
- nome;
- status;
- domínio;
- marca;
- modelo.

### Nível 2: comparação de consultas

Faça três buscas independentes:

```text
Smart TV 55
Smart TV 65
Smart TV 70
```

Crie uma única base contendo uma coluna `termo_busca` que identifique a origem de cada registro. Compare a quantidade de produtos retornados e as marcas mais frequentes.

### Nível 3: paginação

Modifique a função `buscar_produtos_ml()` para coletar até três páginas de resultados para `Smartphone 256 GB`. Verifique duplicidades pelo `id`.

### Nível 4: enriquecimento

Selecione cinco `product_id` e consulte `/products/{PRODUCT_ID}`. Crie uma base com preço da oferta vencedora, moeda, frete grátis e permalink quando esses campos estiverem disponíveis.

### Nível 5: desafio analítico

Escolha duas categorias de produtos de consumo. Construa um pequeno relatório comparando:

- quantidade de produtos encontrados;
- principais marcas;
- disponibilidade de preço no `buy_box_winner`;
- frequência de frete grátis;
- valores ausentes por campo.

Explique quais limitações impedem interpretar o resultado como uma amostra estatisticamente representativa de todo o marketplace.


### Respostas

### Nível 1: busca simples

Pesquise `notebook gamer` no catálogo brasileiro e construa um `DataFrame` com pelo menos:

- `id`;
- nome;
- status;
- domínio;
- marca;
- modelo.

In [ ]:
df_notebook_gamer = buscar_produtos_ml("notebook gamer", paginas=3)
df_notebook_gamer.head(2)

,id,catalog_product_id,domain_id,name,children_ids,attributes,...,settings.content,settings.listing_strategy,settings.exclusive,short_description.type,short_description.content,parent_id
0,MLB52846446,MLB52846446,MLB-NOTEBOOKS,Notebook gamer Victus preto - AMD,[],"[{'id': 'BRAND', 'name': 'Marca', 'value_id': ...",...,fixed,open,False,plaintext,,NaN
1,MLB25959779,MLB25959779,MLB-HOME_OFFICE_DESKS,Mesa Computador Escrivaninha Notebook Gamer Br...,[],"[{'id': 'BRAND', 'name': 'Marca', 'value_id': ...",...,fixed,catalog_required,False,plaintext,"Seja para estudar ou trabalhar, esta mesa Loja...",MLB25602980


### Exercicio 2
Faça três buscas independentes:

```text
Smart TV 55
Smart TV 65
Smart TV 70
```

Crie uma única base contendo uma coluna `termo_busca` que identifique a origem de cada registro. Compare a quantidade de produtos retornados e as marcas mais frequentes.

In [ ]:
df_tv_55_multiplas_paginas = buscar_produtos_ml("Smart TV 55", paginas=3)
df_tv_65_multiplas_paginas = buscar_produtos_ml("Smart TV 65", paginas=3)
df_tv_70_multiplas_paginas = buscar_produtos_ml("Smart TV 70", paginas=3)

df_tv_55_multiplas_paginas["termo_busca"] = "Smart TV 55"
df_tv_65_multiplas_paginas["termo_busca"] = "Smart TV 65"
df_tv_70_multiplas_paginas["termo_busca"] = "Smart TV 70"

resultado = pd.concat([df_tv_55_multiplas_paginas, df_tv_65_multiplas_paginas, df_tv_70_multiplas_paginas], ignore_index=True)
print(resultado)





              id catalog_product_id        domain_id  \
0    MLB37510675        MLB37510675  MLB-TELEVISIONS   
1    MLB79317562        MLB79317562  MLB-TELEVISIONS   
2    MLB79318529        MLB79318529  MLB-TELEVISIONS   
3    MLB30396436        MLB30396436  MLB-TELEVISIONS   
4    MLB52318955        MLB52318955  MLB-TELEVISIONS   
..           ...                ...              ...   
175  MLB67463262        MLB67463262  MLB-TELEVISIONS   
176  MLB29041152        MLB29041152  MLB-TELEVISIONS   
177  MLB37164476        MLB37164476  MLB-TELEVISIONS   
178  MLB52855166        MLB52855166  MLB-TELEVISIONS   
179  MLB61647740        MLB61647740  MLB-TELEVISIONS   

                                                  name children_ids  \
0    Smart TV Kalley 55 GTV55UHDQv2 4K UHD QLED Sma...           []   
1    Samsung Smart Tv 55 4k U8000h + Samsung Smart ...           []   
2    Samsung Smart Tv 55 4k U8000h + Smart Tv 43 Ql...           []   
3    Smart TV de 55" CCE SMART TV 4K CCE CC

### Nível 3: paginação

Modifique a função `buscar_produtos_ml()` para coletar até três páginas de resultados para `Smartphone 256 GB`. Verifique duplicidades pelo `id`.

In [ ]:
df_smartphone_256 = buscar_produtos_ml("Smartphone 256 GB", paginas=3)
df_smartphone= df_smartphone_256.drop_duplicates(subset=["id"], keep="first")
df_smartphone.head(10)

,id,catalog_product_id,domain_id,name,parent_id,children_ids,...,settings.content,settings.listing_strategy,settings.exclusive,short_description.type,short_description.content,enhanced_content_template_id
0,MLB61495414,MLB61495414,MLB-CELLPHONES,Xiaomi Smartphone Poco M7 Pro 5g 256/8gb -glob...,MLB50205748,[],...,fixed,catalog_required,False,plaintext,,NaN
1,MLB45802228,MLB45802228,MLB-CELLPHONES,Smartphone Honor X7c 4g 256 Gb Memória 8Gb Ram...,MLB45801999,[],...,fixed,catalog_required,False,plaintext,O Honor X7C oferece desempenho excepcional com...,NaN
2,MLB33389798,MLB33389798,MLB-CELLPHONES,Smartphone Samsung Galaxy A25 5g 6.5 Polegadas...,MLB31498594,[],...,fixed,catalog_required,False,plaintext,,4d2129b6-1a71-45cc-8e28-ecf1dadf9908
3,MLB33389795,MLB33389795,MLB-CELLPHONES,Smartphone Samsung Galaxy A25 5g 6.5 Polegadas...,MLB31498594,[],...,fixed,catalog_required,False,plaintext,,4d2129b6-1a71-45cc-8e28-ecf1dadf9908
4,MLB47314211,MLB47314211,MLB-CELLPHONES,Smartphone Redmi Note 14 Dual Sim 256/8gb 2025...,MLB45818959,[],...,fixed,catalog_required,False,plaintext,,NaN
5,MLB47311983,MLB47311983,MLB-CELLPHONES,Smartphone Redmi Note 14 Dual Sim 256/8gb 2025...,MLB47309242,[],...,fixed,catalog_required,False,plaintext,,NaN
6,MLB67400477,MLB67400477,MLB-CELLPHONES,Motorola Moto Edge 70 5g Telefone Dual Sim 256...,MLB67400456,[],...,fixed,catalog_required,False,plaintext,,NaN
7,MLB74356746,MLB74356746,MLB-CELLPHONES,Celular Smartphone Realme 14 5g Dual Sim 256 G...,MLB74356742,[],...,fixed,catalog_required,False,plaintext,O Realme 14 5G é a combinação perfeita entre d...,NaN
8,MLB67400457,MLB67400457,MLB-CELLPHONES,Motorola Moto Edge 70 5g Telefone Dual Sim 256...,MLB67400456,[],...,fixed,catalog_required,False,plaintext,,NaN
9,MLB54147310,MLB54147310,MLB-CELLPHONES,Telefone Pepe K800 Android 13 Dual Sim de 10 G...,MLB54147308,[],...,fixed,catalog_required,False,plaintext,----------------- APRESENTAÇÃO DO PRODUTO ----...,NaN


### Nível 4: enriquecimento

Selecione cinco `product_id` e consulte `/products/{PRODUCT_ID}`. Crie uma base com preço da oferta vencedora, moeda, frete grátis e permalink quando esses campos estiverem disponíveis.

In [ ]:
ids_selecionados = df_smartphone["id"].sample(5).tolist()

produtos = []

for id in ids_selecionados:
    resposta = requisicao_get(
        f"https://api.mercadolibre.com/products/{id}",
        headers=headers_ml
    )
    produtos.append(resposta.json())

Status HTTP: 200
Status HTTP: 200
Status HTTP: 200
Status HTTP: 200
Status HTTP: 200


### Nível 5: desafio analítico

Escolha duas categorias de produtos de consumo. Construa um pequeno relatório comparando:

- quantidade de produtos encontrados;
- principais marcas;
- disponibilidade de preço no `buy_box_winner`;
- frequência de frete grátis;
- valores ausentes por campo.

Explique quais limitações impedem interpretar o resultado como uma amostra estatisticamente representativa de todo o marketplace.

In [ ]:

categorias = {
    "Smartphone": df_smartphone,
    "Tênis": df_tenis
}

detalhes = []

for categoria, df in categorias.items():
    ids = df["id"].dropna().drop_duplicates().head(10)

    for product_id in ids:
        resposta = requisicao_get(f"https://api.mercadolibre.com/products/{product_id}", headers=headers_ml)
        produto = resposta.json()
        vencedor = produto.get("buy_box_winner") or {}

        detalhes.append({
            "categoria": categoria,
            "product_id": produto.get("id"),
            "nome": produto.get("name"),
            "preco": vencedor.get("price"),
            "moeda": vencedor.get("currency_id"),
            "frete_gratis": (vencedor.get("shipping") or {}).get("free_shipping")
        })

df_relatorio = pd.DataFrame(detalhes)

resumo = df_relatorio.groupby("categoria").agg(
    produtos=("product_id", "nunique"),
    precos_disponiveis=("preco", "count"),
    frete_gratis=("frete_gratis", "sum")
).reset_index()

print("Resumo comparativo:")
display(resumo)

print("Detalhes coletados:")
display(df_relatorio)